# ตรวจใบกะเพรา ดี / เสีย (Binary Classification, MobileNetV2 + Grad-CAM)

**ก่อนรัน**
1. Runtime > Change runtime type > **T4 GPU**
2. โหลด dataset Good/Bad จาก Mendeley แล้วอัปโหลดเป็น .zip ไว้ใน Google Drive
3. แก้ `ZIP_PATH` ในเซลล์ที่ 2 แล้วรันจากบนลงล่าง

ลำดับ: เตรียมข้อมูล → เช็กสถิติแยกคลาส → แบ่ง train/val/test → ฝึก → ประเมิน → ตัวอย่างถูก/ผิด → Grad-CAM → ทดสอบข้อมูลที่ไม่เคยเห็น

In [ ]:
!pip -q install grad-cam
import os, random, hashlib, zipfile, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1) โหลดข้อมูล

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/basil/holy_basil_good_bad.zip"   # <- แก้ให้ตรงกับไฟล์ของคุณ
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

# ดูโครงสร้างโฟลเดอร์ (เช็กว่าชื่อโฟลเดอร์คลาสคืออะไร)
for root, dirs, files in os.walk(DATA_DIR):
    n = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if n:
        print(root, n)

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png')
CLASSES = ['bad', 'good']        # bad = 0, good = 1

def label_from_name(name, strict=True):
    # map ชื่อโฟลเดอร์ -> label (ถ้าชื่อโฟลเดอร์ไม่ตรง ให้แก้ keyword ตรงนี้)
    n = name.lower()
    bad_kw = ['bad', 'unhealthy', 'disease', 'infect', 'blight', 'mildew', 'spot', 'yellow', 'fusarium', 'powdery']
    good_kw = ['good', 'healthy']
    if any(k in n for k in bad_kw): return 0
    if any(k in n for k in good_kw): return 1
    return None if strict else 0

def collect_images(root, strict=True):
    rows, unknown = [], set()
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(IMG_EXT):
                lab = label_from_name(os.path.basename(r), strict)
                if lab is None:
                    unknown.add(os.path.basename(r)); continue
                rows.append((os.path.join(r, f), lab))
    if unknown:
        print("โฟลเดอร์ที่ map ไม่ได้ (ข้ามไป):", unknown)
    return pd.DataFrame(rows, columns=['path', 'label'])

def load_rgb(p, max_side=640):
    im = Image.open(p)
    im.draft('RGB', (max_side, max_side))   # เร็วขึ้นกับ JPEG ภาพใหญ่
    return im.convert('RGB')

df = collect_images(DATA_DIR)
print(df.label.map({0: 'bad', 1: 'good'}).value_counts())

# ตัดภาพซ้ำแบบเหมือนกันทุกไบต์ (กัน data leakage)
def md5(p):
    with open(p, 'rb') as f: return hashlib.md5(f.read()).hexdigest()
df['hash'] = df.path.map(md5)
conflict = (df.groupby('hash').label.nunique() > 1).sum()
before = len(df)
df = df.drop_duplicates('hash').reset_index(drop=True)
print(f"ตัดภาพซ้ำ {before - len(df)} ภาพ เหลือ {len(df)} | ภาพเดียวกันแต่ label ขัดกัน: {conflict}")

## 2) เช็กสถิติภาพแยกคลาส (กันโมเดลเรียนรู้ทางลัด)
ถ้า ขนาด/ความสว่าง/ขนาดไฟล์ ของสองคลาสต่างกันชัด โมเดลอาจใช้สิ่งเหล่านี้แทนสภาพใบ

In [ ]:
def img_stats(p):
    with Image.open(p) as im:
        w, h = im.size
    g = np.asarray(load_rgb(p, 256).convert('L').resize((64, 64)), dtype=np.float32)
    return w, h, g.mean(), os.path.getsize(p) / 1024

df[['w', 'h', 'brightness', 'kb']] = pd.DataFrame(df.path.map(img_stats).tolist(), index=df.index)
df['class'] = df.label.map({0: 'bad', 1: 'good'})
print(df.groupby('class')[['w', 'h', 'brightness', 'kb']].agg(['mean', 'std']).round(1))

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, c in enumerate(['bad', 'good']):
    sub = df[df['class'] == c].sample(8, random_state=SEED)
    for j, p in enumerate(sub.path):
        axes[i][j].imshow(load_rgb(p)); axes[i][j].axis('off'); axes[i][j].set_title(c)
plt.show()

## 3) แบ่ง train / val / test (แบ่งก่อน augment) 70/15/15

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(n, len(d), d.label.value_counts().sort_index().to_dict())
# หมายเหตุ: ถ้าใบเดียวกันถ่ายหลายมุมและชื่อไฟล์บอกได้ ควรแบ่งตามใบ ไม่ใช่ตามภาพ

## 4a) Background augmentation (สำคัญ)
ภาพใน dataset ถ่ายบนพื้นเทาเข้มเหมือนกันหมด โมเดลจึงอาจเรียนรู้พื้นหลังแทนสภาพใบ (เจอตอน Grad-CAM ชี้ไปที่พื้นหลัง)
แก้โดยตัดใบออกจากพื้นหลัง (ใช้ความอิ่มสี + Otsu) แล้วสุ่มเปลี่ยนพื้นหลังระหว่างฝึก

In [ ]:
import cv2

def leaf_mask(pil):
    # พื้นหลังเทาเข้ม = อิ่มสีต่ำ, ใบ (แม้เหลืองซีด) อิ่มสีสูงกว่า -> แยกด้วย Otsu
    rgb = np.asarray(pil.convert('RGB'))
    s = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[..., 1]
    th, _ = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (s > th).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return np.full(mask.shape, 255, np.uint8)
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, [max(cnts, key=cv2.contourArea)], -1, 255, -1)   # เติมรูในใบ ไม่ให้จุดโรคหาย
    return cv2.GaussianBlur(filled, (5, 5), 0)

def random_bg(h, w):
    t = random.random()
    if t < 0.30:                                   # สีเรียบ (ขาว เทา ดำ สีอื่น)
        c = np.random.randint(0, 256, 3) if random.random() < 0.5 else np.repeat(np.random.randint(0, 256), 3)
        return np.tile(c.astype(np.uint8), (h, w, 1))
    if t < 0.50:                                   # ไล่เฉดสี
        a, b = np.random.randint(0, 256, 3), np.random.randint(0, 256, 3)
        g = np.linspace(0, 1, h)[:, None, None]
        return np.repeat(a * (1 - g) + b * g, w, axis=1).astype(np.uint8)
    if t < 0.65:                                   # ลายหมากรุก (ภาพ "โปร่งใสปลอม" จากเว็บ)
        s = random.choice([16, 24, 32, 48])
        yy, xx = np.indices((h, w))
        chk = (((yy // s) + (xx // s)) % 2)[..., None]
        c1, c2 = random.choice([((255, 255, 255), (204, 204, 204)), ((255, 255, 255), (230, 230, 230)), ((40, 40, 40), (70, 70, 70))])
        return np.where(chk == 0, np.array(c1, np.uint8), np.array(c2, np.uint8))
    n = np.random.rand(h // 40 + 2, w // 40 + 2, 3).astype(np.float32)   # ก้อนสีเบลอ คล้ายพื้นหลังธรรมชาติ
    bg = cv2.resize(n, (w, h), interpolation=cv2.INTER_CUBIC)
    return (np.clip(bg * np.random.uniform(0.5, 1.0, 3), 0, 1) * 255).astype(np.uint8)

def swap_background(pil):
    rgb = np.asarray(pil.convert('RGB')).astype(np.float32)
    m = leaf_mask(pil)
    ratio = (m > 127).mean()
    if ratio < 0.03 or ratio > 0.97:               # แยกใบไม่ได้ -> ใช้ภาพเดิม
        return pil
    h, w = m.shape
    a = m[..., None].astype(np.float32) / 255
    out = rgb * a + random_bg(h, w).astype(np.float32) * (1 - a)
    return Image.fromarray(out.astype(np.uint8))

# ตรวจด้วยตา: ใบต้องไม่ขาด และจุดดำบนใบเสียต้องยังอยู่ (ถ้ามาสก์พัง ปรับ morphology/threshold ด้านบน)
fig, axes = plt.subplots(2, 8, figsize=(20, 5.5))
for i, lab in enumerate([0, 1]):
    sub = train_df[train_df.label == lab].sample(8, random_state=SEED)
    for j, p in enumerate(sub.path):
        axes[i][j].imshow(swap_background(load_rgb(p))); axes[i][j].axis('off')
    axes[i][0].set_title(CLASSES[lab], loc='left')
plt.show()

## 4) Transform + DataLoader

In [ ]:
IMG = 384   # เดิม 224: รอยตำหนิเล็กอาจหายตอนย่อภาพ
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((int(IMG * 1.15), int(IMG * 1.15))),
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1, hue=0),  # ไม่เปลี่ยน hue เพราะสีเหลืองคือสัญญาณใบเสีย
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),  # กันโมเดลพึ่งความคมของภาพ
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class LeafDS(Dataset):
    def __init__(self, d, tf, p_bg=0.0, seed=None):
        self.paths, self.labels, self.tf = d.path.tolist(), d.label.tolist(), tf
        self.p_bg, self.seed = p_bg, seed
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = load_rgb(self.paths[i])
        if self.seed is not None:                  # ตอนประเมิน ให้พื้นหลังสุ่มแบบซ้ำได้
            random.seed(self.seed + i); np.random.seed(self.seed + i)
        if random.random() < self.p_bg:
            im = swap_background(im)
        return self.tf(im), self.labels[i]

train_dl = DataLoader(LeafDS(train_df, train_tf, p_bg=0.6), batch_size=32, shuffle=True, num_workers=2)
# val: ผสมพื้นหลังเดิม/พื้นหลังสุ่ม 50% เพื่อเลือกโมเดลที่ทนต่อพื้นหลังด้วย
val_dl = DataLoader(LeafDS(val_df, eval_tf, p_bg=0.5, seed=1), batch_size=32, num_workers=2)
test_dl = DataLoader(LeafDS(test_df, eval_tf), batch_size=32, num_workers=2)

## 5) โมเดล MobileNetV2 (transfer learning) และฝึก 2 ระยะ
ระยะ 1 ฝึกเฉพาะหัวโมเดล, ระยะ 2 fine-tune ทั้งโมเดลด้วย learning rate ต่ำ

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.last_channel, 2))
model = model.to(device)

counts = train_df.label.value_counts().sort_index().values
class_w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_w)   # ชดเชยคลาสไม่สมดุล

def run_epoch(dl, train, opt=None):
    model.train(train)
    tot_loss = tot_ok = n = 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(y); tot_ok += (out.argmax(1) == y).sum().item(); n += len(y)
    return tot_loss / n, tot_ok / n

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
BEST_PATH = '/content/best_basil_mobilenetv2_bgaug.pt'

def fit(epochs, lr, tag):
    global best_val_loss
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    for ep in range(epochs):
        tl, ta = run_epoch(train_dl, True, opt)
        vl, va = run_epoch(val_dl, False)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        if vl < best_val_loss:
            best_val_loss = vl
            torch.save({'state_dict': model.state_dict(), 'classes': CLASSES, 'img_size': IMG}, BEST_PATH)
        print(f"[{tag}] ep {ep + 1}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")

for p in model.features.parameters(): p.requires_grad = False
fit(5, 1e-3, 'head')
for p in model.features.parameters(): p.requires_grad = True
fit(10, 1e-4, 'finetune')

In [ ]:
# กราฟ Loss / Accuracy (ใช้ในสไลด์หัวข้อ 4)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history['train_acc'], label='train'); ax[1].plot(history['val_acc'], label='val'); ax[1].set_title('Accuracy'); ax[1].legend()
for a in ax: a.set_xlabel('epoch')
plt.show()

# โหลดโมเดลที่ val loss ดีที่สุด และสำรองไว้ใน Drive
ckpt = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ckpt['state_dict']); model.eval()
os.makedirs('/content/drive/MyDrive/basil', exist_ok=True)
shutil.copy(BEST_PATH, '/content/drive/MyDrive/basil/')
print("บันทึกโมเดลแล้ว")

## 6) ประเมินบน Test set (ใช้ test เท่านั้น ไม่ใช้ val)

In [ ]:
@torch.no_grad()
def predict_df(d, p_bg=0.0):
    dl = DataLoader(LeafDS(d, eval_tf, p_bg=p_bg, seed=0 if p_bg > 0 else None), batch_size=32, num_workers=2)
    probs = [torch.softmax(model(x.to(device)), 1).cpu().numpy() for x, _ in dl]
    probs = np.concatenate(probs)
    return probs.argmax(1), probs

def report(d, title, p_bg=0.0):
    y_true = d.label.values
    y_pred, probs = predict_df(d, p_bg)
    print(f"=== {title} (n={len(d)}) ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print(classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASSES, digits=3, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4)); ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.show()
    return y_pred, probs

test_pred, test_probs = report(test_df, 'Test (พื้นหลังเดิม)')
# เปลี่ยนพื้นหลังทั้งชุด: วัดความทนต่อพื้นหลัง แต่เป็นพื้นหลังสังเคราะห์แบบเดียวกับตอนฝึก
# จึงมองดีเกินจริงได้ ให้ยึดผลจากภาพจริงในหัวข้อ 8 เป็นหลัก
_ = report(test_df, 'Test (เปลี่ยนพื้นหลังทั้งหมด)', p_bg=1.0)

In [ ]:
def grid(rows, title):
    if len(rows) == 0:
        print(title, ': ไม่มี'); return
    fig, axes = plt.subplots(1, len(rows), figsize=(3 * len(rows), 3.6), squeeze=False)
    for ax, (_, r) in zip(axes[0], rows.iterrows()):
        ax.imshow(load_rgb(r.path)); ax.axis('off')
        ax.set_title(f"true: {CLASSES[r.label]}\npred: {CLASSES[r.pred]} ({r.conf:.0%})", fontsize=9)
    fig.suptitle(title); plt.show()

def show_examples(d, pred, probs, n=6):
    d = d.copy(); d['pred'] = pred; d['conf'] = probs.max(1)
    ok = d[d.pred == d.label]; wrong = d[d.pred != d.label]
    grid(ok.sample(min(n, len(ok)), random_state=SEED), 'ทำนายถูก')
    grid(wrong.head(n), 'ทำนายผิด')

show_examples(test_df, test_pred, test_probs)

# ดูว่าตั้ง threshold "ไม่แน่ใจ" เท่าไรดี (ใช้ในแอป)
conf = test_probs.max(1); correct = (test_pred == test_df.label.values)
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    if m.any():
        print(f"threshold {t:.1f}: ตอบ {m.mean():.0%} ของภาพ, แม่น {correct[m].mean():.3f}")

## 7) Grad-CAM (โมเดลมองตรงไหนของใบ)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

cam = GradCAM(model=model, target_layers=[model.features[-1]])

def gradcam_grid(d, n=6):
    sample = d.sample(min(n, len(d)), random_state=SEED)
    fig, axes = plt.subplots(2, len(sample), figsize=(3 * len(sample), 6.2), squeeze=False)
    for k, (_, r) in enumerate(sample.iterrows()):
        pil = load_rgb(r.path)
        im = pil.resize((IMG, IMG))
        x = eval_tf(pil).unsqueeze(0).to(device)
        with torch.no_grad():
            p = torch.softmax(model(x), 1)[0]
        heat = cam(input_tensor=x)[0]
        overlay = show_cam_on_image(np.asarray(im, dtype=np.float32) / 255, heat, use_rgb=True)
        axes[0][k].imshow(im); axes[0][k].axis('off'); axes[0][k].set_title(f"true: {CLASSES[r.label]}")
        axes[1][k].imshow(overlay); axes[1][k].axis('off')
        axes[1][k].set_title(f"pred: {CLASSES[int(p.argmax())]} ({p.max():.0%})")
    plt.show()

gradcam_grid(test_df[test_df.label == 0])   # ใบเสีย: heatmap ควรชี้ที่จุด/รอย
gradcam_grid(test_df[test_df.label == 1])   # ใบดี

## 7b) ตรวจสมมติฐาน: โมเดลพึ่งส่วนล่างของภาพ (ก้านใบ/พื้นด้านล่าง) หรือเปล่า
ลบส่วนล่าง 35% ของภาพเป็นสีเทา แล้วดูว่า P(good) เปลี่ยนมากไหม ถ้าเปลี่ยนมากแปลว่าโมเดลพึ่งส่วนนั้น

In [ ]:
def occlude_bottom(pil, frac=0.35):
    im = np.asarray(pil.resize((IMG, IMG))).copy()
    im[int(IMG * (1 - frac)):] = 70
    return Image.fromarray(im)

@torch.no_grad()
def p_good(pil):
    return torch.softmax(model(eval_tf(pil).unsqueeze(0).to(device)), 1)[0, 1].item()

for _, r in test_df.sample(12, random_state=SEED).iterrows():
    pil = load_rgb(r.path)
    print(f"true {CLASSES[r.label]:4s} | P(good) ปกติ {p_good(pil):.2f} -> ลบส่วนล่าง {p_good(occlude_bottom(pil)):.2f}")

## 8) ทดสอบกับข้อมูลที่โมเดลไม่เคยเห็น (ตรงข้อกำหนดข้อ 6)
จัดโฟลเดอร์ละชุดแบบมีโฟลเดอร์ย่อย `good/` และ `bad/` แล้วแก้ path ด้านล่าง
- `my_photos_plain`: ใบกะเพราที่ถ่ายเอง วางบนพื้นเรียบสีเข้ม
- `my_photos_natural`: ถ่ายธรรมชาติ พื้นหลังรก
- `tulsi_medicinal`: ชุด Medicinal Plant (ดึงเฉพาะโฟลเดอร์ Tulsi_*) โฟลเดอร์ที่มีคำว่า healthy = ดี ที่เหลือ = เสีย

In [ ]:
EXT_SETS = {
    'my_photos_plain':   '/content/drive/MyDrive/basil/my_photos_plain',
    'my_photos_natural': '/content/drive/MyDrive/basil/my_photos_natural',
    'tulsi_medicinal':   '/content/drive/MyDrive/basil/tulsi_medicinal',
}
for name, root in EXT_SETS.items():
    if not os.path.isdir(root):
        print("ข้าม", name, "(ไม่พบโฟลเดอร์)"); continue
    ext = collect_images(root, strict=False)
    if len(ext) == 0:
        print("ข้าม", name, "(ไม่มีภาพ)"); continue
    pred, probs = report(ext, f'External: {name}')
    show_examples(ext, pred, probs, n=5)